In [ ]:
import numpy as np
import pandas as pd
import os
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.modeling import models, fitting
import matplotlib.pyplot as plt
from astroquery.simbad import Simbad
from netCDF4 import Dataset
from astropy.visualization import simple_norm

In [ ]:
def locate_pointsource(map,x0_f,y0_f,vis=False):
    x0_f, y0_f = int(np.round(x0_f)), int(np.round(y0_f))
    map = map[y0_f-20:y0_f+20,x0_f-20:x0_f+20]
    amp = np.max(map) - 200
    y0, x0 = np.unravel_index(map.argmax(), map.shape)
    gauss = models.Gaussian2D(amplitude=amp, x_mean=x0, y_mean=y0, 
                              x_stddev=5, y_stddev=5, theta=0) + models.Const2D(amplitude=200)
    gauss.y_stddev_0.tied = lambda model: model.x_stddev_0
    fitter = fitting.LevMarLSQFitter(calc_uncertainties=True)
    yi, xi = np.indices(map.shape)
    fit_func = fitter(gauss, xi, yi, map)
    x = fit_func.x_mean_0 + x0_f - 20
    y = fit_func.y_mean_0 + y0_f - 20
    if vis:
        print(fit_func)
        plt.imshow(map,origin='lower')
        plt.axvline(20,c='orange',label='Real Location')
        plt.axhline(20,c='orange')
        plt.axvline(fit_func.x_mean_0.value,c='blue',label='Fit Location')
        plt.axhline(fit_func.y_mean_0.value,c='blue')
        plt.legend()
        plt.show()
    return (x, y)

def rot_a(angle):
    m = np.eye(2)
    m[0,0] = m[1,1] = np.cos(angle)
    m[0,1] = +np.sin(angle)
    m[1,0] = -np.sin(angle)
    return m

In [ ]:
path = './testing_files_3/redu00/'#../pointing_maps/cleaned_maps/'#'./testing_files/'#'../pointing_maps/cleaned_maps/'
obs_names = [131947,131949,134710,134712,134714,134859,134861]
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
nc_files = os.listdir('./nc_files')
for obs in obs_names:
    print(obs)
    path_obj = path + str(obs) + '/raw/'
    fits_file_ = fits.open(path_obj + f'toltec_commissioning_a1100_science_{str(obs)}_citlali.fits')
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    nc_file_path = [file for file in nc_files if str(obs) in file][0]
    nc_file = Dataset('./nc_files/' + nc_file_path)
    Tel_PA = nc_file.variables['Data.TelescopeBackend.ActParAng'][:].data # in radians
    Tel_PA = np.median(Tel_PA)
    coord_pix = WCS(map_header,naxis=2).world_to_pixel(real_coordinate)
    cut_map = np.clip(map_dat, a_min=200, a_max=1000)
    map_x, map_y = locate_pointsource(cut_map, coord_pix[0], coord_pix[1],vis=False)
    map_coordinate = WCS(map_header,naxis=2).pixel_to_world(map_x, map_y).icrs # in ICRS
    delta_ra = map_coordinate.ra.arcsec - real_coordinate.ra.arcsec
    delta_dec = map_coordinate.dec.arcsec - real_coordinate.dec.arcsec
    delta_az, delta_alt = rot_a(Tel_PA) @ np.array([-delta_ra, delta_dec])
    print('')
    print('RA/DEC:', -delta_ra, -delta_dec)
    print('Az/Alt:', -delta_az, -delta_alt)
    print('')

In [ ]:
# computed with CCW rotation offset
path = './testing_files_4/redu01/'#../pointing_maps/cleaned_maps/'#'./testing_files/'#'../pointing_maps/cleaned_maps/'
obs_names = [131947,131949,134710,134712,134714,134859,134861]
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
nc_files = os.listdir('./nc_files')
for obs in obs_names:
    print(obs)
    path_obj = path + str(obs) + '/raw/'
    fits_file_ = fits.open(path_obj + f'toltec_commissioning_a1100_science_{str(obs)}_citlali.fits')
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    nc_file_path = [file for file in nc_files if str(obs) in file][0]
    nc_file = Dataset('./nc_files/' + nc_file_path)
    Tel_PA = nc_file.variables['Data.TelescopeBackend.ActParAng'][:].data # in radians
    Tel_PA = np.median(Tel_PA)
    coord_pix = WCS(map_header,naxis=2).world_to_pixel(real_coordinate)
    cut_map = np.clip(map_dat, a_min=200, a_max=1000)
    map_x, map_y = locate_pointsource(cut_map, coord_pix[0], coord_pix[1],vis=True)
    map_coordinate = WCS(map_header,naxis=2).pixel_to_world(map_x, map_y).icrs # in ICRS
    delta_ra = map_coordinate.ra.arcsec - real_coordinate.ra.arcsec
    delta_dec = map_coordinate.dec.arcsec - real_coordinate.dec.arcsec
    delta_az, delta_alt = rot_a(Tel_PA) @ np.array([-delta_ra, delta_dec])
    print('RA/DEC:', -delta_ra, -delta_dec)
    print('Az/Alt:', -delta_az, -delta_alt)
    print('')

In [ ]:
# computed with CW rotation offset
path = './testing_files_5/redu02/'#../pointing_maps/cleaned_maps/'#'./testing_files/'#'../pointing_maps/cleaned_maps/'
obs_names = [131947,131949,134710,134712,134714,134859,134861]
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
nc_files = os.listdir('./nc_files')
for obs in obs_names:
    print(obs)
    path_obj = path + str(obs) + '/raw/'
    fits_file_ = fits.open(path_obj + f'toltec_commissioning_a1100_science_{str(obs)}_citlali.fits')
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    nc_file_path = [file for file in nc_files if str(obs) in file][0]
    nc_file = Dataset('./nc_files/' + nc_file_path)
    Tel_PA = nc_file.variables['Data.TelescopeBackend.ActParAng'][:].data # in radians
    Tel_PA = np.median(Tel_PA)
    coord_pix = WCS(map_header,naxis=2).world_to_pixel(real_coordinate)
    cut_map = np.clip(map_dat, a_min=200, a_max=1000)
    map_x, map_y = locate_pointsource(cut_map, coord_pix[0], coord_pix[1],vis=True)
    map_coordinate = WCS(map_header,naxis=2).pixel_to_world(map_x, map_y).icrs # in ICRS
    delta_ra = map_coordinate.ra.arcsec - real_coordinate.ra.arcsec
    delta_dec = map_coordinate.dec.arcsec - real_coordinate.dec.arcsec
    delta_az, delta_alt = rot_a(Tel_PA) @ np.array([-delta_ra, delta_dec])
    print('RA/DEC:', -delta_ra, -delta_dec)
    print('Az/Alt:', -delta_az, -delta_alt)
    print('')

### Compare Coadds

In [ ]:
# computed with CW rotation offset
path_cal = './cal_offset/toltec_commissioning_a1100_citlali.fits'#'../fits_files/new/signal_comp_2/toltec_commissioning_a1100_citlali.fits'
path_0 = './testing_files_3/redu00/131947/raw/toltec_commissioning_a1100_science_131947_citlali.fits'
path_ccw = './testing_files_4/redu01/131947/raw/toltec_commissioning_a1100_science_131947_citlali.fits'
path_cw = './testing_files_5/redu02/131947/raw/toltec_commissioning_a1100_science_131947_citlali.fits'
paths = [path_cal, path_0, path_ccw, path_cw]
titles = ['Calibration\nOffset', 'No Offset', 'CCW', 'CW']
ps = 30
print('VLA 1623')
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(1,4,figsize=(12,6),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
for i,file in enumerate(paths):
    fits_file_ = fits.open(file)
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    viswcs = WCS(map_header,naxis=2)
    coord_pix = viswcs.world_to_pixel(real_coordinate)
    x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
    map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
    print(map_dat.min())
    print(map_dat.max())
    norm = simple_norm(map_dat, vmin=-28,vmax=300)
    im = axs[i].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
    axs[i].grid(True, linestyle='--', alpha=0.6)
    axs[i].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8)
    axs[i].set_title(titles[i])
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    print(titles[i] + ' peak flux: ' + str(round(np.max(map_dat))))
cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.45, pad=0.03)
name = r'$MJy\,sr^{-1}$'
cbar.set_label(name, labelpad=3, fontsize=12)
cbar.ax.tick_params(labelsize=8)
plt.show()
print('[CHG85] GSS 30 IRS 3')
simbad_results = Simbad.query_object('[CHG85] GSS 30 IRS 3')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(1,4,figsize=(12,6),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
for i,file in enumerate(paths):
    fits_file_ = fits.open(file)
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    viswcs = WCS(map_header,naxis=2)
    coord_pix = viswcs.world_to_pixel(real_coordinate)
    x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
    map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
    print(map_dat.min())
    print(map_dat.max())
    norm = simple_norm(map_dat, vmin=-16,vmax=54)
    im = axs[i].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
    axs[i].grid(True, linestyle='--', alpha=0.6)
    axs[i].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8)
    axs[i].set_title(titles[i])
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    print(titles[i] + ' peak flux: ' + str(round(np.max(map_dat))))
cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.45, pad=0.03)
name = r'$MJy\,sr^{-1}$'
cbar.set_label(name, labelpad=3, fontsize=12)
cbar.ax.tick_params(labelsize=8)
plt.show()
print('[GY92] 21')
simbad_results = Simbad.query_object('[GY92] 21')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(1,4,figsize=(12,6),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
for i,file in enumerate(paths):
    fits_file_ = fits.open(file)
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    viswcs = WCS(map_header,naxis=2)
    coord_pix = viswcs.world_to_pixel(real_coordinate)
    x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
    map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
    print(map_dat.min())
    print(map_dat.max())
    norm = simple_norm(map_dat, vmin=-41,vmax=17)
    im = axs[i].imshow(map_dat,origin='lower',norm=norm,cmap='twilight_shifted')
    axs[i].grid(True, linestyle='--', alpha=0.6)
    axs[i].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8)
    axs[i].set_title(titles[i])
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    print(titles[i] + ' peak flux: ' + str(round(np.max(map_dat))))
cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.45, pad=0.03)
name = r'$MJy\,sr^{-1}$'
cbar.set_label(name, labelpad=3, fontsize=12)
cbar.ax.tick_params(labelsize=8)
plt.show()

for obs in obs_names:
    path_cw = f'./testing_files_5/redu02/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
    print(fits.open(path_cw)[0].header['EXPTIME']/60)

In [ ]:
# computed with CW rotation offset
titles = ['Calibration\nOffset', 'No Offset', 'CCW', 'CW']
ps = 30
print('VLA 1623')
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(8,4,figsize=(12,24),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
obs_names = ['Coadded', 131947, 131949, 134710, 134712, 134714, 134859, 134861]
minmaxes = [(-100,1050),(-27,301),(-25,280),(-48,413),(-46,390),(-46,341),(-45,514),(-47,543)]
maxes = []
mins = []
for i, obsmaxes in enumerate(zip(obs_names, minmaxes)):
    obs = obsmaxes[0]
    minmax = obsmaxes[1]
    if obs == 'Coadded':
        path_cal = f'./testing_files_og/redu03/{obs}/raw/toltec_commissioning_a1100_citlali.fits'#./cal_offset/toltec_commissioning_a1100_citlali.fits'#'../fits_files/new/signal_comp_2/toltec_commissioning_a1100_citlali.fits'
        path_0 = f'./testing_files_3/redu00/{obs}/raw/toltec_commissioning_a1100_citlali.fits'
        path_ccw = f'./testing_files_4/redu01/{obs}/raw/toltec_commissioning_a1100_citlali.fits'
        path_cw = f'./testing_files_5/redu02/{obs}/raw/toltec_commissioning_a1100_citlali.fits'
    else:
        path_cal = f'./testing_files_og/redu03/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'#./cal_offset/toltec_commissioning_a1100_citlali.fits'#'../fits_files/new/signal_comp_2/toltec_commissioning_a1100_citlali.fits'
        path_0 = f'./testing_files_3/redu00/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
        path_ccw = f'./testing_files_4/redu01/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
        path_cw = f'./testing_files_5/redu02/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
    paths = [path_cal, path_0, path_ccw, path_cw]
    max_ = []
    min_ = []
    for j,file in enumerate(paths):
        fits_file_ = fits.open(file)
        map_dat = fits_file_[6].data[0][0]
        map_header = fits_file_[6].header
        viswcs = WCS(map_header,naxis=2)
        coord_pix = viswcs.world_to_pixel(real_coordinate)
        x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
        map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
        # print(map_dat.min())
        # print(map_dat.max())
        norm = simple_norm(map_dat, vmin=minmax[0],vmax=minmax[1])
        im = axs[i,j].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
        axs[i,j].grid(True, linestyle='--', alpha=0.6)
        axs[i,j].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8,label='True Coordinate')
        axs[i,j].legend(fontsize=6)
        axs[0,j].set_title(titles[j],fontsize=15)
        axs[i,j].set_xticks([])
        axs[i,j].set_yticks([])
        print(obs)
        if j == 0:
            print(titles[j] + ' peak flux: ' + str(round(np.max(map_dat))))
        max_.append(np.max(map_dat))
        min_.append(np.min(map_dat))
    axs[i,0].set_ylabel(obs,fontsize=15)
    cbar = fig.colorbar(im, ax=axs[i,-1], orientation='vertical', shrink=0.8, pad=0.03)
    name = r'$MJy\,Beam^{-1}$'
    cbar.set_label(name, labelpad=3, fontsize=12)
    cbar.ax.tick_params(labelsize=8)
    maxes.append(max_)
    mins.append(min_)
plt.show()


In [ ]:
# computed with CW rotation offset
titles = ['Calibration\nOffset', 'No Offset', 'CCW', 'CW']
ps = 30
print('VLA 1623')
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
obs_names = ['Coadded', 131947, 131949, 134710, 134712, 134714, 134859, 134861]
minmaxes = [(-100,1050),(-27,301),(-25,280),(-48,413),(-46,390),(-46,341),(-45,514),(-47,543)]
maxes = []
mins = []
for i, obsmaxes in enumerate(zip(obs_names, minmaxes)):
    fig, axs = plt.subplots(1,4,figsize=(10,4),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
    obs = obsmaxes[0]
    minmax = obsmaxes[1]
    if obs == 'Coadded':
        path_cal = f'./testing_files_og/redu03/{obs}/raw/toltec_commissioning_a1100_citlali.fits'#./cal_offset/toltec_commissioning_a1100_citlali.fits'#'../fits_files/new/signal_comp_2/toltec_commissioning_a1100_citlali.fits'
        path_0 = f'./testing_files_3/redu00/{obs}/raw/toltec_commissioning_a1100_citlali.fits'
        path_ccw = f'./testing_files_4/redu01/{obs}/raw/toltec_commissioning_a1100_citlali.fits'
        path_cw = f'./testing_files_5/redu02/{obs}/raw/toltec_commissioning_a1100_citlali.fits'
    else:
        path_cal = f'./testing_files_og/redu03/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'#./cal_offset/toltec_commissioning_a1100_citlali.fits'#'../fits_files/new/signal_comp_2/toltec_commissioning_a1100_citlali.fits'
        path_0 = f'./testing_files_3/redu00/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
        path_ccw = f'./testing_files_4/redu01/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
        path_cw = f'./testing_files_5/redu02/{obs}/raw/toltec_commissioning_a1100_science_{obs}_citlali.fits'
    paths = [path_cal, path_0, path_ccw, path_cw]
    max_ = []
    min_ = []
    for j,file in enumerate(paths):
        fits_file_ = fits.open(file)
        map_dat = fits_file_[6].data[0][0]
        map_header = fits_file_[6].header
        viswcs = WCS(map_header,naxis=2)
        coord_pix = viswcs.world_to_pixel(real_coordinate)
        x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
        map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
        # print(map_dat.min())
        # print(map_dat.max())
        norm = simple_norm(map_dat, vmin=minmax[0],vmax=minmax[1])
        im = axs[j].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
        axs[j].grid(True, linestyle='--', alpha=0.6)
        axs[j].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8,label='True Coordinate')
        axs[j].legend(fontsize=6)
        axs[j].set_title(titles[j],fontsize=10)
        axs[j].set_xticks([])
        axs[j].set_yticks([])
        #print(titles[j] + ' peak flux: ' + str(round(np.max(map_dat))))
        max_.append(np.max(map_dat))
        min_.append(np.min(map_dat))
    axs[0].set_ylabel(obs,fontsize=10)
    cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.5, pad=0.03)
    name = r'$MJy\,Beam^{-1}$'
    cbar.set_label(name, labelpad=3, fontsize=12)
    cbar.ax.tick_params(labelsize=8) 
    maxes.append(max_)
    mins.append(min_)
    plt.show()


### Compare Maps

In [ ]:
# computed with CW rotation offset
path_cal = './cal_offset/toltec_commissioning_a1100_citlali.fits'#'../fits_files/new/signal_comp_2/toltec_commissioning_a1100_citlali.fits'
path_0 = './testing_files_3/redu00/coadded/raw/toltec_commissioning_a1100_citlali.fits'
path_ccw = './testing_files_4/redu01/coadded/raw/toltec_commissioning_a1100_citlali.fits'
path_cw = './testing_files_5/redu02/coadded/raw/toltec_commissioning_a1100_citlali.fits'
paths = [path_cal, path_0, path_ccw, path_cw]
titles = ['Calibration\nOffset', 'No Offset', 'CCW', 'CW']
ps = 30
print('VLA 1623')
simbad_results = Simbad.query_object('VLA 1623')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(1,4,figsize=(12,6),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
for i,file in enumerate(paths):
    fits_file_ = fits.open(file)
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    viswcs = WCS(map_header,naxis=2)
    coord_pix = viswcs.world_to_pixel(real_coordinate)
    x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
    map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
    norm = simple_norm(map_dat, vmin=-120,vmax=1100)
    im = axs[i].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
    axs[i].grid(True, linestyle='--', alpha=0.6)
    axs[i].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8)
    axs[i].set_title(titles[i])
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    print(titles[i] + ' peak flux: ' + str(round(np.max(map_dat))))
cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.45, pad=0.03)
name = r'$MJy\,sr^{-1}$'
cbar.set_label(name, labelpad=3, fontsize=12)
cbar.ax.tick_params(labelsize=8)
plt.show()
print('[CHG85] GSS 30 IRS 3')
simbad_results = Simbad.query_object('[CHG85] GSS 30 IRS 3')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(1,4,figsize=(12,6),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
for i,file in enumerate(paths):
    fits_file_ = fits.open(file)
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    viswcs = WCS(map_header,naxis=2)
    coord_pix = viswcs.world_to_pixel(real_coordinate)
    x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
    map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
    norm = simple_norm(map_dat, vmin=-60,vmax=180)
    im = axs[i].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
    axs[i].grid(True, linestyle='--', alpha=0.6)
    axs[i].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8)
    axs[i].set_title(titles[i])
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    print(titles[i] + ' peak flux: ' + str(round(np.max(map_dat))))
cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.45, pad=0.03)
name = r'$MJy\,sr^{-1}$'
cbar.set_label(name, labelpad=3, fontsize=12)
cbar.ax.tick_params(labelsize=8)
plt.show()
print('[GY92] 21')
simbad_results = Simbad.query_object('[GY92] 21')
RA, DEC = simbad_results.to_pandas()[['ra','dec']].values[0]
real_coordinate = SkyCoord(ra=RA, dec=DEC, frame='icrs', obstime="J2000", unit=(u.deg,u.deg))
fig, axs = plt.subplots(1,4,figsize=(12,6),sharex=True,sharey=True,constrained_layout=True)#,subplot_kw=dict(projection=viswcs))
for i,file in enumerate(paths):
    fits_file_ = fits.open(file)
    map_dat = fits_file_[6].data[0][0]
    map_header = fits_file_[6].header
    viswcs = WCS(map_header,naxis=2)
    coord_pix = viswcs.world_to_pixel(real_coordinate)
    x0_f, y0_f = int(np.round(coord_pix[0])), int(np.round(coord_pix[1]))
    map_dat = map_dat[y0_f-ps//2:y0_f+ps//2,x0_f-ps//2:x0_f+ps//2]
    norm = simple_norm(map_dat, vmin=-130,vmax=130)
    im = axs[i].imshow(map_dat,origin='lower',norm=norm,cmap='twilight')
    axs[i].grid(True, linestyle='--', alpha=0.6)
    axs[i].scatter(x=ps//2,y=ps//2,marker='x',linewidth=2,color='grey',alpha=0.8)
    axs[i].set_title(titles[i])
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    print(titles[i] + ' peak flux: ' + str(round(np.max(map_dat))))
cbar = fig.colorbar(im, ax=axs[-1], orientation='vertical', shrink=0.45, pad=0.03)
name = r'$MJy\,sr^{-1}$'
cbar.set_label(name, labelpad=3, fontsize=12)
cbar.ax.tick_params(labelsize=8)
plt.show()

## Put into table

In [ ]:
from astropy.table import QTable
import numpy as np
from astropy.io import ascii
import io

offsets_az = [2.59, 2.58, -2.83, -2.18, 3.97, 1.18, 0.19]
offsets_el = [2.66, 2.11, -0.53, 2.44, 2.97, -3.53, -3.22]
obs_names = [131947, 131949, 134710, 134712, 134714, 134859, 134861]
best_offsets = QTable(data=np.array([obs_names,offsets_az,offsets_el]).T, names=['Observation', 'Az Offset', 'El Offset'], dtype=[int,float,float])
output_stream = io.StringIO()
ascii.write(best_offsets, './pointing_table', format='latex')
latex_string = output_stream.getvalue()